In [1]:
# ==========================================
# TINYLLAMA ENVIRONMENT DIAGNOSTIC
# NO MODEL
# NO DATASET
# NO GENERATION
# ==========================================

import sys
import os
import torch

print("=" * 70)
print("PYTHON / PYTORCH DIAGNOSTIC")
print("=" * 70)

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():

    print("CUDA version:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))

    props = torch.cuda.get_device_properties(0)

    print(
        "VRAM:",
        round(props.total_memory / 1024**3, 2),
        "GB"
    )

    print(
        "Allocated:",
        round(torch.cuda.memory_allocated() / 1024**3, 2),
        "GB"
    )

    print(
        "Reserved:",
        round(torch.cuda.memory_reserved() / 1024**3, 2),
        "GB"
    )

print("=" * 70)
print("SYSTEM MEMORY")
print("=" * 70)

try:
    import psutil

    ram = psutil.virtual_memory()

    print(
        "RAM total:",
        round(ram.total / 1024**3, 2),
        "GB"
    )

    print(
        "RAM available:",
        round(ram.available / 1024**3, 2),
        "GB"
    )

    print(
        "RAM used:",
        round(ram.used / 1024**3, 2),
        "GB"
    )

except Exception as e:
    print("RAM check unavailable:", e)

PYTHON / PYTORCH DIAGNOSTIC
Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
PyTorch: 2.6.0+cu124
CUDA available: True
CUDA version: 12.4
GPU: NVIDIA GeForce RTX 3060 Laptop GPU
VRAM: 6.0 GB
Allocated: 0.0 GB
Reserved: 0.0 GB
SYSTEM MEMORY
RAM total: 15.7 GB
RAM available: 5.78 GB
RAM used: 9.92 GB


In [1]:
# ==========================================
# TINYLLAMA ENVIRONMENT DIAGNOSTIC
# NO MODEL
# NO DATASET
# NO GENERATION
# ==========================================

import sys
import os
import torch

print("=" * 70)
print("PYTHON / PYTORCH DIAGNOSTIC")
print("=" * 70)

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():

    print("CUDA version:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))

    props = torch.cuda.get_device_properties(0)

    print(
        "VRAM:",
        round(props.total_memory / 1024**3, 2),
        "GB"
    )

    print(
        "Allocated:",
        round(torch.cuda.memory_allocated() / 1024**3, 2),
        "GB"
    )

    print(
        "Reserved:",
        round(torch.cuda.memory_reserved() / 1024**3, 2),
        "GB"
    )

print("=" * 70)
print("SYSTEM MEMORY")
print("=" * 70)

try:
    import psutil

    ram = psutil.virtual_memory()

    print(
        "RAM total:",
        round(ram.total / 1024**3, 2),
        "GB"
    )

    print(
        "RAM available:",
        round(ram.available / 1024**3, 2),
        "GB"
    )

    print(
        "RAM used:",
        round(ram.used / 1024**3, 2),
        "GB"
    )

except Exception as e:
    print("RAM check unavailable:", e)

PYTHON / PYTORCH DIAGNOSTIC
Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
PyTorch: 2.6.0+cu124
CUDA available: True
CUDA version: 12.4
GPU: NVIDIA GeForce RTX 3060 Laptop GPU
VRAM: 6.0 GB
Allocated: 0.0 GB
Reserved: 0.0 GB
SYSTEM MEMORY
RAM total: 15.7 GB
RAM available: 5.85 GB
RAM used: 9.85 GB


In [2]:
# ============================================================
# TINYLLAMA — ASSET 2,359 SAMPLE EVALUATION
# ZERO-SHOT / FEW-SHOT / CoT
# INFERENCE ONLY — NO TRAINING
# ============================================================

import os
import gc
import time
import torch
import pandas as pd
import numpy as np

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import evaluate
import textstat

print("=" * 80)
print("TINYLLAMA ASSET EVALUATION")
print("=" * 80)

# ============================================================
# CONFIGURATION
# ============================================================

CHECKPOINT = r"C:\Users\mohsi\tinyllama-plainlang-rtx3060\checkpoint-8000"

OUTPUT_DIR = r"C:\Users\mohsi\tinyllama-asset-evaluation"

os.makedirs(OUTPUT_DIR, exist_ok=True)

ZERO_SHOT_CSV = os.path.join(
    OUTPUT_DIR,
    "TINYLLAMA_ZERO_SHOT_2359.csv"
)

FEW_SHOT_CSV = os.path.join(
    OUTPUT_DIR,
    "TINYLLAMA_FEW_SHOT_2359.csv"
)

COT_CSV = os.path.join(
    OUTPUT_DIR,
    "TINYLLAMA_COT_2359.csv"
)

BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

MAX_NEW_TOKENS = 128

print("\nCheckpoint:")
print(CHECKPOINT)

print("\nOutput directory:")
print(OUTPUT_DIR)

# ============================================================
# GPU
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required for this evaluation.")

device = torch.device("cuda")

print("\nGPU:")
print(torch.cuda.get_device_name(0))

print(
    "VRAM:",
    round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3,
        2
    ),
    "GB"
)

# ============================================================
# LOAD TOKENIZER
# ============================================================

print("\nLoading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    use_fast=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded.")

# ============================================================
# LOAD BASE MODEL
# ============================================================

print("\nLoading TinyLlama base model...")

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

# ============================================================
# LOAD LoRA CHECKPOINT
# ============================================================

print("\nLoading LoRA checkpoint...")

model = PeftModel.from_pretrained(
    base_model,
    CHECKPOINT
)

model = model.merge_and_unload()

model = model.to(device)

model.eval()

print("TinyLlama checkpoint loaded successfully.")

# ============================================================
# MEMORY CLEANUP
# ============================================================

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

print(
    "\nCurrent VRAM:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

# ============================================================
# LOAD ASSET
# ============================================================

print("\nLoading ASSET...")

asset = load_dataset("facebook/asset")

print(asset)

for split in asset.keys():
    print(
        f"{split}: {len(asset[split])} samples"
    )

# ============================================================
# SELECT 2,359 SAMPLES
# ============================================================

# ASSET evaluation split
asset_split = asset["test"]

N_SAMPLES = min(2359, len(asset_split))

asset_data = asset_split.select(
    range(N_SAMPLES)
)

print("\nSelected samples:", len(asset_data))

# ============================================================
# INSPECT DATA FORMAT
# ============================================================

print("\nFirst ASSET example:")
print(asset_data[0])

# ============================================================
# LOAD METRICS
# ============================================================

print("\nLoading metrics...")

sari_metric = evaluate.load("sari")
bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")

print("Metrics loaded.")

print("\n" + "=" * 80)
print("SETUP COMPLETE")
print("=" * 80)
print("No training performed.")
print("No optimizer loaded.")
print("No gradients enabled.")
print("Ready for evaluation.")

TINYLLAMA ASSET EVALUATION

Checkpoint:
C:\Users\mohsi\tinyllama-plainlang-rtx3060\checkpoint-8000

Output directory:
C:\Users\mohsi\tinyllama-asset-evaluation

GPU:
NVIDIA GeForce RTX 3060 Laptop GPU
VRAM: 6.0 GB

Loading tokenizer...
Tokenizer loaded.

Loading TinyLlama base model...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Loading LoRA checkpoint...


ValueError: Can't find 'adapter_config.json' at 'C:\Users\mohsi\tinyllama-plainlang-rtx3060\checkpoint-8000'

In [3]:
# ==========================================
# FIND TINYLLAMA LORA CHECKPOINT
# ==========================================

import os

ROOT = r"C:\Users\mohsi\tinyllama-plainlang-rtx3060"

print("=" * 80)
print("SEARCHING FOR LORA ADAPTER CHECKPOINTS")
print("=" * 80)

if not os.path.exists(ROOT):
    raise FileNotFoundError(f"Folder not found: {ROOT}")

for item in sorted(os.listdir(ROOT)):

    path = os.path.join(ROOT, item)

    if os.path.isdir(path) and item.startswith("checkpoint-"):

        files = os.listdir(path)

        print("\n" + "-" * 80)
        print(item)

        if "adapter_config.json" in files:
            print("✅ adapter_config.json FOUND")

        if "adapter_model.safetensors" in files:
            print("✅ adapter_model.safetensors FOUND")

        if "adapter_model.bin" in files:
            print("✅ adapter_model.bin FOUND")

        if "model.safetensors" in files:
            print("⚠️ model.safetensors FOUND")

        print("Files:")
        for f in sorted(files):
            print("   ", f)

SEARCHING FOR LORA ADAPTER CHECKPOINTS

--------------------------------------------------------------------------------
checkpoint-4300
✅ adapter_config.json FOUND
✅ adapter_model.safetensors FOUND
Files:
    README.md
    adapter_config.json
    adapter_model.safetensors
    optimizer.pt
    rng_state.pth
    scheduler.pt
    trainer_state.json
    training_args.bin


In [4]:
# ============================================================
# LOAD FINE-TUNED TINYLLAMA — CHECKPOINT 4300
# INFERENCE ONLY — NO TRAINING
# ============================================================

import os
import gc
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

CHECKPOINT = (
    r"C:\Users\mohsi\tinyllama-plainlang-rtx3060"
    r"\checkpoint-4300"
)

print("=" * 80)
print("LOADING FINE-TUNED TINYLLAMA")
print("=" * 80)

print("\nBase model:")
print(BASE_MODEL)

print("\nLoRA checkpoint:")
print(CHECKPOINT)

# ------------------------------------------------------------
# Verify checkpoint
# ------------------------------------------------------------

assert os.path.exists(CHECKPOINT), (
    f"Checkpoint not found:\n{CHECKPOINT}"
)

assert os.path.exists(
    os.path.join(CHECKPOINT, "adapter_config.json")
), "adapter_config.json not found."

assert os.path.exists(
    os.path.join(CHECKPOINT, "adapter_model.safetensors")
), "adapter_model.safetensors not found."

print("\n✅ LoRA checkpoint verified.")

# ------------------------------------------------------------
# Tokenizer
# ------------------------------------------------------------

print("\nLoading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    use_fast=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ------------------------------------------------------------
# Base model
# ------------------------------------------------------------

print("Loading base TinyLlama...")

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch.float16,
    low_cpu_mem_usage=True
)

# ------------------------------------------------------------
# LoRA adapter
# ------------------------------------------------------------

print("Loading LoRA adapter...")

model = PeftModel.from_pretrained(
    base_model,
    CHECKPOINT,
    is_trainable=False
)

# ------------------------------------------------------------
# Merge adapter into base model
# ------------------------------------------------------------

print("Merging LoRA adapter...")

model = model.merge_and_unload()

# ------------------------------------------------------------
# GPU
# ------------------------------------------------------------

model = model.to("cuda")
model.eval()

# ------------------------------------------------------------
# Cleanup
# ------------------------------------------------------------

gc.collect()
torch.cuda.empty_cache()

print("\n" + "=" * 80)
print("✅ FINE-TUNED TINYLLAMA READY")
print("=" * 80)

print(
    "GPU:",
    torch.cuda.get_device_name(0)
)

print(
    "VRAM allocated:",
    round(
        torch.cuda.memory_allocated() / 1024**3,
        2
    ),
    "GB"
)

print("\nTraining:")
print("❌ No training")
print("❌ No optimizer")
print("❌ No gradients")

print("\nMode:")
print("✅ Evaluation / inference only")
print("✅ LoRA checkpoint-4300 loaded")

LOADING FINE-TUNED TINYLLAMA

Base model:
TinyLlama/TinyLlama-1.1B-Chat-v1.0

LoRA checkpoint:
C:\Users\mohsi\tinyllama-plainlang-rtx3060\checkpoint-4300

✅ LoRA checkpoint verified.

Loading tokenizer...
Loading base TinyLlama...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading LoRA adapter...
Merging LoRA adapter...

✅ FINE-TUNED TINYLLAMA READY
GPU: NVIDIA GeForce RTX 3060 Laptop GPU
VRAM allocated: 2.05 GB

Training:
❌ No training
❌ No optimizer
❌ No gradients

Mode:
✅ Evaluation / inference only
✅ LoRA checkpoint-4300 loaded


In [5]:
# ============================================================
# LOAD ASSET + EVALUATION METRICS
# FINE-TUNED TINYLLAMA — CHECKPOINT 4300
# ============================================================

import os
import gc
import pandas as pd
import numpy as np
import evaluate
import textstat

from datasets import load_dataset

print("=" * 80)
print("LOADING ASSET + METRICS")
print("=" * 80)

# ============================================================
# OUTPUT DIRECTORY
# ============================================================

OUTPUT_DIR = r"C:\Users\mohsi\tinyllama-asset-evaluation"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

print("\nOutput directory:")
print(OUTPUT_DIR)

# ============================================================
# LOAD ASSET
# ============================================================

print("\nLoading ASSET...")

asset = load_dataset("facebook/asset")

print("\nDataset loaded:")
print(asset)

for split in asset.keys():
    print(
        f"{split}: {len(asset[split])} samples"
    )

# ============================================================
# SELECT TEST SET
# ============================================================

asset_test = asset["test"]

N_SAMPLES = min(
    2359,
    len(asset_test)
)

asset_eval = asset_test.select(
    range(N_SAMPLES)
)

print("\n" + "=" * 80)
print("EVALUATION DATA")
print("=" * 80)

print("Samples selected:", len(asset_eval))

print("\nFirst sample:")
print(asset_eval[0])

# ============================================================
# LOAD METRICS
# ============================================================

print("\n" + "=" * 80)
print("LOADING METRICS")
print("=" * 80)

print("\nLoading SARI...")
sari_metric = evaluate.load("sari")

print("Loading BLEU...")
bleu_metric = evaluate.load("bleu")

print("Loading ROUGE...")
rouge_metric = evaluate.load("rouge")

print("Loading BERTScore...")
bertscore_metric = evaluate.load("bertscore")

print("\n✅ All metrics loaded.")

# ============================================================
# MEMORY CLEANUP
# ============================================================

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n" + "=" * 80)
print("✅ ASSET + METRICS READY")
print("=" * 80)

print("Fine-tuned model : checkpoint-4300")
print("ASSET samples    :", len(asset_eval))
print("Metrics          : SARI / BLEU / ROUGE-L / BERTScore / FKGL")
print("Generation       : NOT STARTED")

LOADING ASSET + METRICS

Output directory:
C:\Users\mohsi\tinyllama-asset-evaluation

Loading ASSET...


'[Errno 11002] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/datasets/facebook/asset/resolve/c7f2fa4bae55ae656091805d4416c1374582bb4e/asset.py
Retrying in 1s [Retry 1/5].
Using the latest cached version of the dataset since facebook/asset couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'simplification' at C:\Users\mohsi\.cache\huggingface\datasets\facebook___asset\simplification\0.0.0\c7f2fa4bae55ae656091805d4416c1374582bb4e (last modified on Mon May 25 05:44:58 2026).


ValueError: Feature type 'List' not found. Available feature types: ['Value', 'ClassLabel', 'Translation', 'TranslationVariableLanguages', 'LargeList', 'Sequence', 'Array2D', 'Array3D', 'Array4D', 'Array5D', 'Audio', 'Image', 'Video', 'Pdf']

In [6]:
# ==========================================
# FIND LOCAL ASSET CACHE
# ==========================================

import os

ASSET_CACHE = os.path.expanduser(
    r"~\.cache\huggingface\datasets\facebook___asset"
)

print("=" * 80)
print("ASSET CACHE")
print("=" * 80)

print("Path:")
print(ASSET_CACHE)

print("\nExists:", os.path.exists(ASSET_CACHE))

if os.path.exists(ASSET_CACHE):

    for root, dirs, files in os.walk(ASSET_CACHE):

        level = root.replace(ASSET_CACHE, "").count(os.sep)

        if level > 3:
            dirs[:] = []

        indent = "    " * level

        print(f"{indent}{os.path.basename(root)}/")

        for file in files[:20]:
            print(f"{indent}    {file}")

ASSET CACHE
Path:
C:\Users\mohsi\.cache\huggingface\datasets\facebook___asset

Exists: True
facebook___asset/
    simplification/
        0.0.0/
            c7f2fa4bae55ae656091805d4416c1374582bb4e/
                asset-test.arrow
                asset-validation.arrow
                dataset_info.json


In [7]:
# ============================================================
# LOAD CACHED ASSET DIRECTLY
# BYPASS HUGGING FACE DATASET BUILDER
# ============================================================

from datasets import Dataset
import os

ASSET_CACHE = os.path.expanduser(
    r"~\.cache\huggingface\datasets\facebook___asset"
)

ASSET_VERSION_DIR = os.path.join(
    ASSET_CACHE,
    "simplification",
    "0.0.0",
    "c7f2fa4bae55ae656091805d4416c1374582bb4e"
)

ASSET_TEST_ARROW = os.path.join(
    ASSET_VERSION_DIR,
    "asset-test.arrow"
)

print("=" * 80)
print("LOADING CACHED ASSET TEST SET")
print("=" * 80)

print("\nArrow file:")
print(ASSET_TEST_ARROW)

if not os.path.exists(ASSET_TEST_ARROW):
    raise FileNotFoundError(
        f"ASSET test file not found:\n{ASSET_TEST_ARROW}"
    )

# ------------------------------------------------------------
# Load Arrow directly
# ------------------------------------------------------------

asset_test = Dataset.from_file(
    ASSET_TEST_ARROW
)

print("\n✅ ASSET test set loaded directly from Arrow.")

print("\nNumber of samples:")
print(len(asset_test))

print("\nColumns:")
print(asset_test.column_names)

print("\nFirst sample:")
print(asset_test[0])

# ------------------------------------------------------------
# Select exactly 2,359
# ------------------------------------------------------------

N_SAMPLES = min(
    2359,
    len(asset_test)
)

asset_eval = asset_test.select(
    range(N_SAMPLES)
)

print("\n" + "=" * 80)
print("ASSET EVALUATION SET READY")
print("=" * 80)

print("Total cached test samples :", len(asset_test))
print("Selected evaluation samples:", len(asset_eval))

print("\nFirst evaluation sample:")
print(asset_eval[0])

LOADING CACHED ASSET TEST SET

Arrow file:
C:\Users\mohsi\.cache\huggingface\datasets\facebook___asset\simplification\0.0.0\c7f2fa4bae55ae656091805d4416c1374582bb4e\asset-test.arrow


ValueError: Feature type 'List' not found. Available feature types: ['Value', 'ClassLabel', 'Translation', 'TranslationVariableLanguages', 'LargeList', 'Sequence', 'Array2D', 'Array3D', 'Array4D', 'Array5D', 'Audio', 'Image', 'Video', 'Pdf']

In [8]:
# ============================================================
# LOAD ASSET ARROW DIRECTLY WITH PYARROW
# BYPASS HUGGING FACE DATASETS METADATA
# ============================================================

import os
import pyarrow as pa
import pyarrow.ipc as ipc
import pandas as pd

ASSET_TEST_ARROW = os.path.expanduser(
    r"~\.cache\huggingface\datasets\facebook___asset"
    r"\simplification\0.0.0"
    r"\c7f2fa4bae55ae656091805d4416c1374582bb4e"
    r"\asset-test.arrow"
)

print("=" * 80)
print("READING ASSET ARROW DIRECTLY")
print("=" * 80)

print("\nFile:")
print(ASSET_TEST_ARROW)

if not os.path.exists(ASSET_TEST_ARROW):
    raise FileNotFoundError(ASSET_TEST_ARROW)

# ------------------------------------------------------------
# Open Arrow file WITHOUT Hugging Face Dataset
# ------------------------------------------------------------

with pa.memory_map(ASSET_TEST_ARROW, "r") as source:
    reader = ipc.open_stream(source)
    table = reader.read_all()

print("\n✅ Arrow file read successfully.")

print("\nRows:", table.num_rows)
print("Columns:", table.column_names)

print("\nSchema:")
print(table.schema)

# ------------------------------------------------------------
# Convert to Python/Pandas
# ------------------------------------------------------------

asset_df = table.to_pandas()

print("\nDataFrame shape:")
print(asset_df.shape)

print("\nDataFrame columns:")
print(asset_df.columns.tolist())

print("\nFirst sample:")
print(asset_df.iloc[0].to_dict())

# ------------------------------------------------------------
# Select exactly 2,359 samples
# ------------------------------------------------------------

N_SAMPLES = min(
    2359,
    len(asset_df)
)

asset_eval_df = asset_df.iloc[:N_SAMPLES].copy()

asset_eval_df.reset_index(
    drop=True,
    inplace=True
)

print("\n" + "=" * 80)
print("ASSET EVALUATION SET READY")
print("=" * 80)

print("Total ASSET rows :", len(asset_df))
print("Selected rows    :", len(asset_eval_df))

print("\nFirst selected sample:")
print(asset_eval_df.iloc[0].to_dict())

READING ASSET ARROW DIRECTLY

File:
C:\Users\mohsi\.cache\huggingface\datasets\facebook___asset\simplification\0.0.0\c7f2fa4bae55ae656091805d4416c1374582bb4e\asset-test.arrow

✅ Arrow file read successfully.

Rows: 359
Columns: ['original', 'simplifications']

Schema:
original: string
simplifications: list<item: string>
  child 0, item: string
-- schema metadata --
huggingface: '{"info": {"features": {"original": {"dtype": "string", "_ty' + 104

DataFrame shape:
(359, 2)

DataFrame columns:
['original', 'simplifications']

First sample:
{'original': 'One side of the armed conflicts is composed mainly of the Sudanese military and the Janjaweed, a Sudanese militia group recruited mostly from the Afro-Arab Abbala tribes of the northern Rizeigat region in Sudan.', 'simplifications': array(['On one side of the conflicts are the Sudanese military and the Janjaweed, a Sudanese militia group.  They are mostly recruited from the Afro-Arab Abbala tribes.',
       'One side of the armed conflicts

In [9]:
# ============================================================
# ASSET — EXACT SAME DATASET AS QWEN
# VALIDATION + TEST = 2,359 SAMPLES
# DIRECT ARROW LOADING
# ============================================================

import os
import pyarrow as pa
import pyarrow.ipc as ipc
import pandas as pd

ASSET_ROOT = os.path.expanduser(
    r"~\.cache\huggingface\datasets\facebook___asset"
    r"\simplification\0.0.0"
    r"\c7f2fa4bae55ae656091805d4416c1374582bb4e"
)

VALIDATION_ARROW = os.path.join(
    ASSET_ROOT,
    "asset-validation.arrow"
)

TEST_ARROW = os.path.join(
    ASSET_ROOT,
    "asset-test.arrow"
)


def load_asset_arrow(path):

    with pa.memory_map(path, "r") as source:

        reader = ipc.open_stream(source)

        table = reader.read_all()

    return table.to_pandas()


print("=" * 100)
print("📂 LOADING ASSET — VALIDATION + TEST")
print("=" * 100)

print("\nValidation file:")
print(VALIDATION_ARROW)

print("\nTest file:")
print(TEST_ARROW)


assert os.path.exists(
    VALIDATION_ARROW
), "Validation Arrow file not found."

assert os.path.exists(
    TEST_ARROW
), "Test Arrow file not found."


# ============================================================
# LOAD BOTH SPLITS
# ============================================================

validation_df = load_asset_arrow(
    VALIDATION_ARROW
)

test_df = load_asset_arrow(
    TEST_ARROW
)


print("\nValidation samples:", len(validation_df))
print("Test samples:", len(test_df))


# ============================================================
# COMBINE — EXACTLY LIKE QWEN
# ============================================================

asset_df = pd.concat(
    [
        validation_df,
        test_df
    ],
    ignore_index=True
)


# ============================================================
# VERIFY
# ============================================================

print("\n" + "=" * 100)
print("🔍 ASSET VERIFICATION")
print("=" * 100)

print("\nUnique source sentences:", len(asset_df))

print(
    "Total human references:",
    asset_df["simplifications"]
    .apply(len)
    .sum()
)

assert len(asset_df) == 2359

assert (
    asset_df["simplifications"]
    .apply(len)
    .eq(10)
    .all()
)

print("\n✅ ASSET verification passed.")
print("Sources:", len(asset_df))

print(
    "References:",
    asset_df["simplifications"]
    .apply(len)
    .sum()
)

print("References per sample: 10")

print("\nFirst sample:")
print(asset_df.iloc[0].to_dict())

📂 LOADING ASSET — VALIDATION + TEST

Validation file:
C:\Users\mohsi\.cache\huggingface\datasets\facebook___asset\simplification\0.0.0\c7f2fa4bae55ae656091805d4416c1374582bb4e\asset-validation.arrow

Test file:
C:\Users\mohsi\.cache\huggingface\datasets\facebook___asset\simplification\0.0.0\c7f2fa4bae55ae656091805d4416c1374582bb4e\asset-test.arrow

Validation samples: 2000
Test samples: 359

🔍 ASSET VERIFICATION

Unique source sentences: 2359
Total human references: 23590

✅ ASSET verification passed.
Sources: 2359
References: 23590
References per sample: 10

First sample:
{'original': 'Adjacent counties are Marin (to the south), Mendocino (to the north), Lake (northeast), Napa (to the east), and Solano and Contra Costa (to the southeast).', 'simplifications': array(['countries next to it are Marin, Mendocino, Lake, Napa, Solano, and Contra Costa.',
       'Nearby counties are Marin, Mendocino, Lake, Napa, and Solano and Contra Costa.',
       'Adjacent counties are Marin, Mendocino, L

In [10]:
# =====================================================================
# TINYLLAMA — EXACT GEMMA/QWEN ZERO-SHOT PROMPT
# =====================================================================
# ⚠️ DO NOT CHANGE THIS PROMPT
# =====================================================================

def zero_shot_prompt(sentence):

    return f"""Simplify the following sentence using clear and easy-to-understand language.

Preserve the original meaning.
Do not add explanations or commentary.
Return only the simplified sentence.

Original sentence:
{sentence}

Simplified sentence:"""


# =====================================================================
# PREPARE TINYLLAMA CHAT INPUT
# =====================================================================
# The PROMPT TEXT is identical to Gemma/Qwen.
# Only TinyLlama's native chat template is used for formatting.
# =====================================================================

def prepare_tinyllama_input(sentence):

    prompt = zero_shot_prompt(sentence)

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    return text


print("=" * 100)
print("📝 TINYLLAMA ZERO-SHOT PROMPT")
print("=" * 100)

print(
    zero_shot_prompt(
        "This is a test sentence."
    )
)

print("\n" + "=" * 100)
print("🔧 TINYLLAMA FORMATTED INPUT")
print("=" * 100)

print(
    prepare_tinyllama_input(
        "This is a test sentence."
    )
)

print("\n✅ Exact Gemma/Qwen zero-shot prompt ready.")
print("✅ TinyLlama chat formatting ready.")

📝 TINYLLAMA ZERO-SHOT PROMPT
Simplify the following sentence using clear and easy-to-understand language.

Preserve the original meaning.
Do not add explanations or commentary.
Return only the simplified sentence.

Original sentence:
This is a test sentence.

Simplified sentence:

🔧 TINYLLAMA FORMATTED INPUT
<|user|>
Simplify the following sentence using clear and easy-to-understand language.

Preserve the original meaning.
Do not add explanations or commentary.
Return only the simplified sentence.

Original sentence:
This is a test sentence.

Simplified sentence:</s>
<|assistant|>


✅ Exact Gemma/Qwen zero-shot prompt ready.
✅ TinyLlama chat formatting ready.


In [12]:
# ============================================================
# RESTORE TINYLLAMA OUTPUT CLEANER
# ============================================================

def clean_tinyllama_output(text):

    stop_markers = [
        "<|endoftext|>",
        "<|user|>",
        "<|assistant|>",
        "</s>",
        "Original sentence:",
        "Simplified sentence:"
    ]

    for marker in stop_markers:
        if marker in text:
            text = text.split(marker)[0]

    return " ".join(text.strip().split())


print("✅ clean_tinyllama_output() restored.")

✅ clean_tinyllama_output() restored.


In [13]:
# =====================================================================
# TINYLLAMA ZERO-SHOT — SINGLE SAMPLE TEST
# SAME GENERATION SETTINGS AS QWEN
# =====================================================================

print("=" * 100)
print("🧪 TINYLLAMA ZERO-SHOT — SINGLE SAMPLE TEST")
print("=" * 100)

# ------------------------------------------------------------
# Take first ASSET sample
# ------------------------------------------------------------

example = asset_df.iloc[0]

source = example["original"]

references = example["simplifications"]


print("\n🔵 ORIGINAL:")
print(source)


# ------------------------------------------------------------
# Prepare EXACT zero-shot prompt
# ------------------------------------------------------------

formatted_prompt = prepare_tinyllama_input(
    source
)


# ------------------------------------------------------------
# Tokenize
# ------------------------------------------------------------

inputs = tokenizer(
    formatted_prompt,
    return_tensors="pt"
).to(model.device)


# ------------------------------------------------------------
# Reset VRAM measurement
# ------------------------------------------------------------

if torch.cuda.is_available():

    torch.cuda.empty_cache()

    torch.cuda.reset_peak_memory_stats()


# ------------------------------------------------------------
# GENERATION
# ------------------------------------------------------------

start_time = time.perf_counter()

with torch.no_grad():

    output_ids = model.generate(

        **inputs,

        max_new_tokens=128,

        do_sample=False,

        use_cache=True,

        pad_token_id=tokenizer.eos_token_id,

        eos_token_id=tokenizer.eos_token_id
    )


end_time = time.perf_counter()


inference_time = (
    end_time - start_time
)


# ------------------------------------------------------------
# REMOVE PROMPT TOKENS
# ------------------------------------------------------------

generated_ids = output_ids[
    0,
    inputs["input_ids"].shape[1]:
]


# ------------------------------------------------------------
# DECODE
# ------------------------------------------------------------

raw_prediction = tokenizer.decode(
    generated_ids,
    skip_special_tokens=False
)


# ------------------------------------------------------------
# CLEAN TINYLLAMA CHAT ARTIFACTS
# ------------------------------------------------------------

prediction = clean_tinyllama_output(
    raw_prediction
)


# ------------------------------------------------------------
# PEAK VRAM
# ------------------------------------------------------------

if torch.cuda.is_available():

    peak_vram = (
        torch.cuda.max_memory_allocated()
        / (1024 ** 3)
    )

else:

    peak_vram = None


# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

print("\n🟢 TINYLLAMA GENERATED:")

print(prediction)


print("\n⚡ PERFORMANCE")

print("-" * 60)

print(
    f"Inference Time: "
    f"{inference_time:.3f} sec"
)

if peak_vram is not None:

    print(
        f"Peak VRAM     : "
        f"{peak_vram:.3f} GB"
    )


print("\n📚 REFERENCES:")

for idx, reference in enumerate(
    references,
    start=1
):

    print(
        f"{idx}. {reference}"
    )


# ------------------------------------------------------------
# CLEAN GPU MEMORY
# ------------------------------------------------------------

del inputs
del output_ids
del generated_ids

gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()


print("\n✅ SINGLE-SAMPLE GENERATION TEST COMPLETE.")
print("⚠️ No 2,359-sample evaluation has started.")

[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🧪 TINYLLAMA ZERO-SHOT — SINGLE SAMPLE TEST

🔵 ORIGINAL:
Adjacent counties are Marin (to the south), Mendocino (to the north), Lake (northeast), Napa (to the east), and Solano and Contra Costa (to the southeast).

🟢 TINYLLAMA GENERATED:
Adjacent counties are Marin (to the south ) , Mendocino (to the north ) , Lake (northeast ) , Napa (to the east ) , and Solano and Contra Costa (to the southeast ) .

⚡ PERFORMANCE
------------------------------------------------------------
Inference Time: 5.151 sec
Peak VRAM     : 2.072 GB

📚 REFERENCES:
1. countries next to it are Marin, Mendocino, Lake, Napa, Solano, and Contra Costa.
2. Nearby counties are Marin, Mendocino, Lake, Napa, and Solano and Contra Costa.
3. Adjacent counties are Marin, Mendocino, Lake, Napa, Solano and Contra Costa.
4. Neighboring counties are Marin, Mendocino, Lake, Napa, Solano, and Contra Costa.
5. Adjacent counties are Marin (south), Mendocino (north), Lake (northeast), and Napa (east). Solano and Contra Costa are to t

In [14]:
# ============================================================
# CLEAN GENERATION CONFIGURATION
# ============================================================

if hasattr(model, "generation_config"):

    model.generation_config.max_length = None

    print("✅ model.generation_config.max_length = None")

print("✅ max_new_tokens=128 will be the generation limit.")
print("✅ do_sample=False")
print("✅ EOS stopping enabled")

✅ model.generation_config.max_length = None
✅ max_new_tokens=128 will be the generation limit.
✅ do_sample=False
✅ EOS stopping enabled


In [15]:
# ============================================================
# TINYLLAMA ZERO-SHOT — SINGLE SAMPLE METRICS TEST
# SAME METRIC PIPELINE AS QWEN / GEMMA
# ============================================================

print("=" * 100)
print("📊 TINYLLAMA ZERO-SHOT — SINGLE SAMPLE METRICS TEST")
print("=" * 100)

scores = evaluate_one_sample(
    source,
    prediction,
    references
)

print("\n📊 METRICS")
print("-" * 60)

print(
    f"SARI          : {scores['sari']:.4f}"
)

print(
    f"BLEU          : {scores['bleu']:.4f}"
)

print(
    f"ROUGE-L       : {scores['rougeL']:.4f}"
)

print(
    f"BERTScore-F1  : {scores['bertscore_f1']:.4f}"
)

print(
    f"FKGL          : {scores['fkgl']:.4f}"
)

print("\n✅ Single-sample metrics calculated.")
print("⚠️ Full 2,359-sample evaluation has NOT started.")

📊 TINYLLAMA ZERO-SHOT — SINGLE SAMPLE METRICS TEST


NameError: name 'evaluate_one_sample' is not defined

In [16]:
# =====================================================================
# TINYLLAMA — SAME EVALUATION METRIC FUNCTION AS QWEN
# =====================================================================

import evaluate
import numpy as np
import textstat

print("=" * 100)
print("📊 LOADING EVALUATION METRICS")
print("=" * 100)

sari_metric = evaluate.load("sari")
bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")

print("\n✅ All metrics loaded successfully.")


# =====================================================================
# EVALUATE ONE SAMPLE
# =====================================================================

def evaluate_one_sample(
    source,
    prediction,
    references
):

    prediction = str(
        prediction
    ).strip()

    # ================================================================
    # EMPTY PREDICTION SAFETY
    # ================================================================

    if not prediction:

        return {
            "sari": 0.0,
            "bleu": 0.0,
            "rougeL": 0.0,
            "bertscore_f1": 0.0,
            "fkgl": 0.0
        }


    # ================================================================
    # SARI — ALL 10 REFERENCES
    # ================================================================

    try:

        sari_result = sari_metric.compute(
            sources=[source],
            predictions=[prediction],
            references=[references]
        )

        sari = sari_result["sari"]

    except Exception:

        sari = 0.0


    # ================================================================
    # BLEU — ALL 10 REFERENCES
    # ================================================================

    try:

        bleu_result = bleu_metric.compute(
            predictions=[prediction],
            references=[references],
            smooth=True
        )

        bleu = bleu_result["bleu"]

    except Exception:

        bleu = 0.0


    # ================================================================
    # ROUGE-L — MEAN OF 10 REFERENCES
    # ================================================================

    rouge_scores = []

    for reference in references:

        try:

            rouge_result = rouge_metric.compute(
                predictions=[prediction],
                references=[reference]
            )

            rouge_scores.append(
                rouge_result["rougeL"]
            )

        except Exception:

            rouge_scores.append(0.0)


    rouge_l = float(
        np.mean(rouge_scores)
    )


    # ================================================================
    # BERTSCORE-F1 — MEAN OF 10 REFERENCES
    # ================================================================

    try:

        bert_result = bertscore_metric.compute(

            predictions=[
                prediction
            ] * len(references),

            references=references,

            lang="en"
        )

        bertscore_f1 = float(
            np.mean(
                bert_result["f1"]
            )
        )

    except Exception:

        bertscore_f1 = 0.0


    # ================================================================
    # FKGL — GENERATED TEXT ONLY
    # ================================================================

    try:

        fkgl = textstat.flesch_kincaid_grade(
            prediction
        )

    except Exception:

        fkgl = 0.0


    # ================================================================
    # RETURN
    # ================================================================

    return {

        "sari": float(sari),

        "bleu": float(bleu),

        "rougeL": float(rouge_l),

        "bertscore_f1": float(
            bertscore_f1
        ),

        "fkgl": float(fkgl)
    }


print("\n✅ evaluate_one_sample() ready.")
print("✅ SARI      → 10 references")
print("✅ BLEU      → 10 references")
print("✅ ROUGE-L   → mean of 10 references")
print("✅ BERTScore → mean of 10 references")
print("✅ FKGL      → generated text only")

📊 LOADING EVALUATION METRICS


Using the latest cached version of the module from C:\Users\mohsi\.cache\huggingface\modules\evaluate_modules\metrics\evaluate-metric--sari\a0dd86b272c088d488925f5735d937ac3f21014d322792ed359597d8a8faaf9b (last modified on Sun Jul  5 02:45:11 2026) since it couldn't be found locally at evaluate-metric--sari, or remotely on the Hugging Face Hub.
Using the latest cached version of the module from C:\Users\mohsi\.cache\huggingface\modules\evaluate_modules\metrics\evaluate-metric--rouge\6e5315f72865c2eaa764c8361360bb938740b9c120a2cf3a7ad218aa0ce452ed (last modified on Sun Jul  5 02:45:24 2026) since it couldn't be found locally at evaluate-metric--rouge, or remotely on the Hugging Face Hub.



✅ All metrics loaded successfully.

✅ evaluate_one_sample() ready.
✅ SARI      → 10 references
✅ BLEU      → 10 references
✅ ROUGE-L   → mean of 10 references
✅ BERTScore → mean of 10 references
✅ FKGL      → generated text only


In [17]:
scores = evaluate_one_sample(
    source,
    prediction,
    references
)

print("\n📊 METRICS")
print("-" * 60)

print(f"SARI          : {scores['sari']:.4f}")
print(f"BLEU          : {scores['bleu']:.4f}")
print(f"ROUGE-L       : {scores['rougeL']:.4f}")
print(f"BERTScore-F1  : {scores['bertscore_f1']:.4f}")
print(f"FKGL          : {scores['fkgl']:.4f}")

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



📊 METRICS
------------------------------------------------------------
SARI          : 53.0608
BLEU          : 1.0000
ROUGE-L       : 0.6602
BERTScore-F1  : 0.9099
FKGL          : 12.5680


In [18]:
# =====================================================================
# 🔥 FINE-TUNED TINYLLAMA — ZERO-SHOT ASSET EVALUATION
# 🔥 2,359 SAMPLES × 10 REFERENCES
# 🔥 EXACT GEMMA/QWEN ZERO-SHOT PROMPT
# 🔥 RESUME-SAFE
# 🔥 NO TRAINING
# =====================================================================

import os
import gc
import time
import numpy as np
import pandas as pd
import torch


# =====================================================================
# CONFIGURATION
# =====================================================================

OUTPUT_DIR = r"C:\Users\mohsi\tinyllama-asset-evaluation"

OUTPUT_FILE = os.path.join(
    OUTPUT_DIR,
    "TINYLLAMA_ZERO_SHOT_ASSET_2359_FINAL.csv"
)

CHECKPOINT_EVERY = 25

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


print("=" * 100)
print("🔥 TINYLLAMA — ZERO-SHOT ASSET EVALUATION")
print("=" * 100)

print("\nCheckpoint:")
print(r"C:\Users\mohsi\tinyllama-plainlang-rtx3060\checkpoint-4300")

print("\nOutput:")
print(OUTPUT_FILE)

print("\nDataset:")
print("ASSET validation + test")

print("\nSamples:")
print(len(asset_df))

print("\nReferences per sample:")
print("10")

print("\nMode:")
print("✅ Evaluation / inference only")
print("❌ No training")
print("❌ No optimizer")
print("❌ No gradients")


# =====================================================================
# VERIFY DATASET
# =====================================================================

assert len(asset_df) == 2359

assert (
    asset_df["simplifications"]
    .apply(len)
    .eq(10)
    .all()
)

print("\n✅ Dataset verification passed.")


# =====================================================================
# VERIFY REQUIRED FUNCTIONS
# =====================================================================

assert callable(
    zero_shot_prompt
), "zero_shot_prompt() is missing."

assert callable(
    prepare_tinyllama_input
), "prepare_tinyllama_input() is missing."

assert callable(
    clean_tinyllama_output
), "clean_tinyllama_output() is missing."

assert callable(
    evaluate_one_sample
), "evaluate_one_sample() is missing."

print("✅ Required functions verified.")


# =====================================================================
# LOAD EXISTING CSV IF PRESENT
# =====================================================================

if os.path.exists(OUTPUT_FILE):

    print("\n" + "=" * 100)
    print("🔄 EXISTING CHECKPOINT FOUND")
    print("=" * 100)

    results_df = pd.read_csv(
        OUTPUT_FILE
    )

    print(
        "\nExisting completed samples:",
        len(results_df)
    )

    if "sample_id" in results_df.columns:

        completed_ids = set(
            results_df[
                "sample_id"
            ].astype(int)
        )

    else:

        completed_ids = set()

        print(
            "⚠️ sample_id column missing."
        )

    print(
        "Already completed:",
        len(completed_ids)
    )

    print(
        "Remaining:",
        len(asset_df) - len(completed_ids)
    )

else:

    print("\n" + "=" * 100)
    print("🆕 NO EXISTING CHECKPOINT")
    print("=" * 100)

    results_df = pd.DataFrame()

    completed_ids = set()

    print(
        "\nStarting from sample 1."
    )


# =====================================================================
# VERIFY MODEL
# =====================================================================

model.eval()

print("\n" + "=" * 100)
print("🤖 MODEL STATUS")
print("=" * 100)

print("\nModel:")
print("Fine-tuned TinyLlama")

print("\nCheckpoint:")
print("checkpoint-4300")

print("\nTraining:")
print("❌ OFF")

print("\nEvaluation:")
print("✅ ON")


# =====================================================================
# MAIN EVALUATION LOOP
# =====================================================================

total_samples = len(asset_df)

start_total_time = time.perf_counter()


for i in range(total_samples):


    # ================================================================
    # RESUME
    # ================================================================

    if i in completed_ids:

        print(
            f"⏭️ SKIPPING SAMPLE "
            f"{i + 1}/{total_samples} "
            f"(already completed)"
        )

        continue


    # ================================================================
    # SAMPLE HEADER
    # ================================================================

    print("\n")
    print("=" * 100)

    print(
        f"🧪 SAMPLE {i + 1}/{total_samples}"
    )

    print("=" * 100)


    # ================================================================
    # DATA
    # ================================================================

    example = asset_df.iloc[i]

    source = str(
        example["original"]
    )

    references = [
        str(x)
        for x in example["simplifications"]
    ]


    # ================================================================
    # ORIGINAL
    # ================================================================

    print("\n🔵 ORIGINAL:")
    print(source)


    # ================================================================
    # EXACT ZERO-SHOT PROMPT
    # ================================================================

    formatted_prompt = prepare_tinyllama_input(
        source
    )


    # ================================================================
    # TOKENIZATION
    # ================================================================

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(model.device)


    # ================================================================
    # VRAM RESET
    # ================================================================

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

        torch.cuda.reset_peak_memory_stats()

        torch.cuda.synchronize()


    # ================================================================
    # GENERATION
    # ================================================================

    start_time = time.perf_counter()


    with torch.no_grad():

        output_ids = model.generate(

            **inputs,

            max_new_tokens=128,

            do_sample=False,

            use_cache=True,

            pad_token_id=tokenizer.eos_token_id,

            eos_token_id=tokenizer.eos_token_id
        )


    if torch.cuda.is_available():

        torch.cuda.synchronize()


    end_time = time.perf_counter()


    inference_time = (
        end_time - start_time
    )


    # ================================================================
    # REMOVE INPUT TOKENS
    # ================================================================

    generated_ids = output_ids[
        0,
        inputs["input_ids"].shape[1]:
    ]


    # ================================================================
    # DECODE
    # ================================================================

    raw_prediction = tokenizer.decode(
        generated_ids,
        skip_special_tokens=False
    )


    # ================================================================
    # CLEAN TINYLLAMA OUTPUT
    # ================================================================

    prediction = clean_tinyllama_output(
        raw_prediction
    )


    # ================================================================
    # PEAK VRAM
    # ================================================================

    if torch.cuda.is_available():

        peak_vram = (
            torch.cuda.max_memory_allocated()
            / (1024 ** 3)
        )

    else:

        peak_vram = None


    # ================================================================
    # DISPLAY GENERATION
    # ================================================================

    print("\n🟢 TINYLLAMA GENERATED:")
    print(prediction)


    # ================================================================
    # METRICS
    # ================================================================

    scores = evaluate_one_sample(

        source,

        prediction,

        references
    )


    # ================================================================
    # DISPLAY METRICS
    # ================================================================

    print("\n📊 METRICS")
    print("-" * 60)

    print(
        f"SARI          : "
        f"{scores['sari']:.4f}"
    )

    print(
        f"BLEU          : "
        f"{scores['bleu']:.4f}"
    )

    print(
        f"ROUGE-L       : "
        f"{scores['rougeL']:.4f}"
    )

    print(
        f"BERTScore-F1  : "
        f"{scores['bertscore_f1']:.4f}"
    )

    print(
        f"FKGL          : "
        f"{scores['fkgl']:.4f}"
    )


    # ================================================================
    # PERFORMANCE
    # ================================================================

    print("\n⚡ PERFORMANCE")
    print("-" * 60)

    print(
        f"Inference Time: "
        f"{inference_time:.3f} sec"
    )

    if peak_vram is not None:

        print(
            f"Peak VRAM     : "
            f"{peak_vram:.3f} GB"
        )


    # ================================================================
    # CREATE RESULT
    # ================================================================

    result = {

        "sample_id": i,

        "model": (
            "TinyLlama-1.1B-Chat-v1.0"
            "-FineTuned-checkpoint-4300"
        ),

        "prompting": "zero-shot",

        "original": source,

        "generated": prediction,

        "references": " ||| ".join(
            references
        ),

        "sari": scores["sari"],

        "bleu": scores["bleu"],

        "rougeL": scores["rougeL"],

        "bertscore_f1": (
            scores["bertscore_f1"]
        ),

        "fkgl": scores["fkgl"],

        "inference_time_sec": (
            inference_time
        ),

        "peak_vram_gb": peak_vram
    }


    # ================================================================
    # APPEND TO DATAFRAME
    # ================================================================

    results_df = pd.concat(

        [
            results_df,

            pd.DataFrame(
                [result]
            )
        ],

        ignore_index=True
    )


    completed_ids.add(i)


    # ================================================================
    # CHECKPOINT
    # ================================================================

    completed_count = len(completed_ids)


    if (

        completed_count % CHECKPOINT_EVERY == 0

        or

        i == total_samples - 1

    ):

        results_df = (
            results_df
            .drop_duplicates(
                subset=["sample_id"],
                keep="last"
            )
            .sort_values(
                "sample_id"
            )
            .reset_index(
                drop=True
            )
        )

        results_df.to_csv(
            OUTPUT_FILE,
            index=False
        )

        print("\n💾 CHECKPOINT SAVED")

        print(
            f"Completed: "
            f"{completed_count}/{total_samples}"
        )

        print(
            "File:",
            OUTPUT_FILE
        )


    # ================================================================
    # PROGRESS
    # ================================================================

    print("\n📈 PROGRESS")

    print(
        f"[{completed_count}/{total_samples}] "
        f"SARI={scores['sari']:.4f} | "
        f"BLEU={scores['bleu']:.4f} | "
        f"ROUGE-L={scores['rougeL']:.4f} | "
        f"BERT={scores['bertscore_f1']:.4f} | "
        f"FKGL={scores['fkgl']:.4f} | "
        f"Time={inference_time:.2f}s | "
        f"VRAM={peak_vram:.3f}GB"
    )


    # ================================================================
    # CLEAN GENERATION OBJECTS
    # ================================================================

    del inputs
    del output_ids
    del generated_ids


    # ================================================================
    # GPU CLEANUP
    # ================================================================

    if completed_count % 10 == 0:

        gc.collect()

        if torch.cuda.is_available():

            torch.cuda.empty_cache()


# =====================================================================
# FINAL SAVE
# =====================================================================

results_df = (
    results_df
    .drop_duplicates(
        subset=["sample_id"],
        keep="last"
    )
    .sort_values(
        "sample_id"
    )
    .reset_index(
        drop=True
    )
)

results_df.to_csv(
    OUTPUT_FILE,
    index=False
)


# =====================================================================
# FINAL SUMMARY
# =====================================================================

total_time = (
    time.perf_counter()
    - start_total_time
)

print("\n")
print("=" * 100)
print("🎉 TINYLLAMA ZERO-SHOT EVALUATION COMPLETE")
print("=" * 100)

print(
    "\nTotal samples:",
    len(results_df)
)

print(
    "\nOutput file:"
)

print(
    OUTPUT_FILE
)


# =====================================================================
# AVERAGES
# =====================================================================

print("\n📊 AVERAGE RESULTS")
print("-" * 60)

print(
    "Average SARI:",
    round(
        results_df["sari"].mean(),
        4
    )
)

print(
    "Average BLEU:",
    round(
        results_df["bleu"].mean(),
        4
    )
)

print(
    "Average ROUGE-L:",
    round(
        results_df["rougeL"].mean(),
        4
    )
)

print(
    "Average BERTScore-F1:",
    round(
        results_df[
            "bertscore_f1"
        ].mean(),
        4
    )
)

print(
    "Average FKGL:",
    round(
        results_df["fkgl"].mean(),
        4
    )
)

print(
    "Average Inference Time:",
    round(
        results_df[
            "inference_time_sec"
        ].mean(),
        4
    ),
    "seconds"
)

print(
    "Average Peak VRAM:",
    round(
        results_df[
            "peak_vram_gb"
        ].mean(),
        4
    ),
    "GB"
)

print(
    "\nTotal wall-clock time:",
    round(
        total_time / 60,
        2
    ),
    "minutes"
)

print("\n✅ ZERO-SHOT DONE.")

🔥 TINYLLAMA — ZERO-SHOT ASSET EVALUATION

Checkpoint:
C:\Users\mohsi\tinyllama-plainlang-rtx3060\checkpoint-4300

Output:
C:\Users\mohsi\tinyllama-asset-evaluation\TINYLLAMA_ZERO_SHOT_ASSET_2359_FINAL.csv

Dataset:
ASSET validation + test

Samples:
2359

References per sample:
10

Mode:
✅ Evaluation / inference only
❌ No training
❌ No optimizer
❌ No gradients

✅ Dataset verification passed.
✅ Required functions verified.

🆕 NO EXISTING CHECKPOINT

Starting from sample 1.

🤖 MODEL STATUS

Model:
Fine-tuned TinyLlama

Checkpoint:
checkpoint-4300

Training:
❌ OFF

Evaluation:
✅ ON


🧪 SAMPLE 1/2359

🔵 ORIGINAL:
Adjacent counties are Marin (to the south), Mendocino (to the north), Lake (northeast), Napa (to the east), and Solano and Contra Costa (to the southeast).

🟢 TINYLLAMA GENERATED:
Adjacent counties are Marin (to the south ) , Mendocino (to the north ) , Lake (northeast ) , Napa (to the east ) , and Solano and Contra Costa (to the southeast ) .

📊 METRICS
---------------------------

In [19]:
# ==========================================
# TINYLLAMA ZERO-SHOT
# DESCRIPTIVE STATISTICS
# MEAN | MEDIAN | MIN | MAX | STD
# ==========================================

import pandas as pd
import numpy as np

INPUT_FILE = r"C:\Users\mohsi\tinyllama-asset-evaluation\TINYLLAMA_ZERO_SHOT_ASSET_2359_FINAL.csv"

# Load results
df = pd.read_csv(INPUT_FILE)

print("=" * 100)
print("📊 TINYLLAMA ZERO-SHOT — DESCRIPTIVE STATISTICS")
print("=" * 100)

print(f"\nTotal Samples: {len(df)}")

print("\nAvailable Columns:")
print(df.columns.tolist())


# ==========================================
# FIND METRIC COLUMNS
# ==========================================

metrics = [
    "sari",
    "bleu",
    "rouge_l",
    "bertscore_f1",
    "fkgl",
    "inference_time",
    "peak_vram"
]

# Keep only columns that actually exist
available_metrics = [col for col in metrics if col in df.columns]

print("\n" + "=" * 100)
print("📈 STATISTICS")
print("=" * 100)

stats = df[available_metrics].agg([
    "mean",
    "median",
    "min",
    "max",
    "std"
]).T

# Better column names
stats.columns = [
    "Mean",
    "Median",
    "Min",
    "Max",
    "Std"
]

# Round for display
stats = stats.round(4)

print("\n")
print(stats.to_string())


# ==========================================
# SAVE STATISTICS
# ==========================================

OUTPUT_STATS = r"C:\Users\mohsi\tinyllama-asset-evaluation\TINYLLAMA_ZERO_SHOT_ASSET_2359_STATISTICS.csv"

stats.to_csv(OUTPUT_STATS)

print("\n" + "=" * 100)
print("💾 STATISTICS SAVED")
print("=" * 100)
print(f"\n{OUTPUT_STATS}")


# ==========================================
# FORMATTED SUMMARY
# ==========================================

print("\n" + "=" * 100)
print("📋 FINAL SUMMARY")
print("=" * 100)

for metric, row in stats.iterrows():
    print(f"\n{metric.upper()}")
    print("-" * 50)
    print(f"Mean   : {row['Mean']:.4f}")
    print(f"Median : {row['Median']:.4f}")
    print(f"Min    : {row['Min']:.4f}")
    print(f"Max    : {row['Max']:.4f}")
    print(f"Std    : {row['Std']:.4f}")

print("\n" + "=" * 100)
print("✅ DESCRIPTIVE STATISTICS COMPLETE")
print("=" * 100)

📊 TINYLLAMA ZERO-SHOT — DESCRIPTIVE STATISTICS

Total Samples: 2359

Available Columns:
['sample_id', 'model', 'prompting', 'original', 'generated', 'references', 'sari', 'bleu', 'rougeL', 'bertscore_f1', 'fkgl', 'inference_time_sec', 'peak_vram_gb']

📈 STATISTICS


                 Mean   Median      Min      Max     Std
sari          49.6039  52.9301  15.6537  77.1272  9.1927
bleu           0.8649   0.9230   0.0350   1.0000  0.1794
bertscore_f1   0.9429   0.9473   0.8007   0.9827  0.0221
fkgl          10.5530  10.4411  -1.4500  28.1089  3.9164

💾 STATISTICS SAVED

C:\Users\mohsi\tinyllama-asset-evaluation\TINYLLAMA_ZERO_SHOT_ASSET_2359_STATISTICS.csv

📋 FINAL SUMMARY

SARI
--------------------------------------------------
Mean   : 49.6039
Median : 52.9301
Min    : 15.6537
Max    : 77.1272
Std    : 9.1927

BLEU
--------------------------------------------------
Mean   : 0.8649
Median : 0.9230
Min    : 0.0350
Max    : 1.0000
Std    : 0.1794

BERTSCORE_F1
------------------------------